# 🔧 Week 3: Data Preprocessing & Feature Engineering

## Overview
Real-world data is messy! This week focuses on the critical skills that separate good ML practitioners from great ones.

## 🎯 Learning Objectives
1. Handle missing values properly
2. Detect and handle outliers
3. Encode categorical variables
4. Scale and normalize features
5. Handle imbalanced datasets
6. **Understand and prevent data leakage**
7. Build reusable preprocessing pipelines

## ⏱️ Estimated Time: 8-10 hours

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    LabelEncoder, OneHotEncoder, OrdinalEncoder
)
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

np.random.seed(42)
print("All imports successful!")

## 1. Create Messy Dataset

Let's create a realistic messy dataset with common issues.

In [ ]:
# Create a messy dataset simulating customer churn prediction
n_samples = 1000

np.random.seed(42)

data = {
    'age': np.random.randint(18, 80, n_samples).astype(float),
    'income': np.random.normal(60000, 25000, n_samples),
    'tenure_months': np.random.randint(1, 72, n_samples),
    'monthly_charges': np.random.uniform(20, 120, n_samples),
    'total_charges': np.random.uniform(100, 8000, n_samples),
    'contract_type': np.random.choice(['Month-to-month', 'One year', 'Two year'], n_samples),
    'payment_method': np.random.choice(['Credit card', 'Bank transfer', 'Electronic check', 'Mailed check'], n_samples),
    'gender': np.random.choice(['Male', 'Female'], n_samples),
    'senior_citizen': np.random.choice([0, 1], n_samples, p=[0.84, 0.16]),
}

# Create target (churn) with some logic
churn_prob = 0.2 + 0.3 * (data['contract_type'] == 'Month-to-month').astype(float)
churn_prob -= 0.1 * (data['tenure_months'] > 24).astype(float)
data['churn'] = (np.random.random(n_samples) < churn_prob).astype(int)

df = pd.DataFrame(data)

# ============================================================
# INTRODUCE DATA QUALITY ISSUES
# ============================================================

# 1. Missing values (random)
for col in ['age', 'income', 'monthly_charges']:
    mask = np.random.random(n_samples) < 0.1  # 10% missing
    df.loc[mask, col] = np.nan

# 2. Missing values with pattern (MNAR - Missing Not At Random)
# Older customers less likely to provide income
age_mask = df['age'] > 60
income_missing_mask = age_mask & (np.random.random(n_samples) < 0.3)
df.loc[income_missing_mask, 'income'] = np.nan

# 3. Outliers
outlier_idx = np.random.choice(n_samples, 10, replace=False)
df.loc[outlier_idx[:5], 'income'] = np.random.uniform(500000, 1000000, 5)
df.loc[outlier_idx[5:], 'age'] = np.random.uniform(-5, 150, 5)

# 4. Inconsistent categorical values
df.loc[np.random.choice(n_samples, 20), 'gender'] = 'male'  # lowercase
df.loc[np.random.choice(n_samples, 15), 'gender'] = 'FEMALE'  # uppercase

print("Dataset shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nFirst few rows:")
df.head()

In [ ]:
# ============================================================
# DATA QUALITY ASSESSMENT
# ============================================================

def data_quality_report(df):
    """Generate a data quality report."""
    report = pd.DataFrame({
        'dtype': df.dtypes,
        'missing': df.isnull().sum(),
        'missing_pct': (df.isnull().sum() / len(df) * 100).round(2),
        'unique': df.nunique(),
        'sample_values': df.apply(lambda x: x.dropna().head(3).tolist())
    })
    return report

print("Data Quality Report:")
print("="*80)
data_quality_report(df)

---
## 2. Handling Missing Values

### 🧠 Interview Concepts

**Types of Missing Data:**
- **MCAR (Missing Completely At Random):** Missingness unrelated to any variable
- **MAR (Missing At Random):** Missingness related to observed variables
- **MNAR (Missing Not At Random):** Missingness related to unobserved values

**Strategies:**
| Strategy | When to Use | Pros | Cons |
|----------|-------------|------|------|
| Drop rows | < 5% missing, MCAR | Simple | Loses data |
| Mean/Median | Numerical, MCAR | Preserves mean | Reduces variance |
| Mode | Categorical | Simple | May skew distribution |
| KNN Imputer | Moderate missing | Considers relationships | Computationally expensive |

In [ ]:
# ============================================================
# MISSING VALUE ANALYSIS
# ============================================================

# Visualize missing values
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar plot of missing values
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
axes[0].bar(missing.index, missing.values)
axes[0].set_title('Missing Values by Column')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Heatmap of missing pattern
missing_cols = df.columns[df.isnull().any()]
sns.heatmap(df[missing_cols].isnull(), cbar=True, ax=axes[1], cmap='viridis')
axes[1].set_title('Missing Value Pattern')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# IMPUTATION STRATEGIES
# ============================================================

# Create a copy for demonstration
df_imputed = df.copy()

# Strategy 1: Simple Imputation (Mean/Median/Mode)
print("Simple Imputation:")
print("="*50)

# Mean for age (numerical)
mean_imputer = SimpleImputer(strategy='mean')
age_imputed = mean_imputer.fit_transform(df_imputed[['age']])
print(f"Age: Mean = {mean_imputer.statistics_[0]:.2f}")

# Median for income (better for skewed data with outliers)
median_imputer = SimpleImputer(strategy='median')
income_imputed = median_imputer.fit_transform(df_imputed[['income']])
print(f"Income: Median = {median_imputer.statistics_[0]:.2f}")

# Strategy 2: KNN Imputation (considers relationships)
print("\nKNN Imputation (uses 5 nearest neighbors):")
knn_imputer = KNNImputer(n_neighbors=5)

# Prepare numerical columns
num_cols = ['age', 'income', 'monthly_charges']
df_knn = df_imputed[num_cols].copy()
df_knn_imputed = pd.DataFrame(
    knn_imputer.fit_transform(df_knn),
    columns=num_cols
)

# Compare imputation results
print("\nComparison of imputed values (first 5 originally missing):")
missing_mask = df['age'].isnull()
comparison = pd.DataFrame({
    'Mean Imputed': age_imputed[missing_mask, 0][:5],
    'KNN Imputed': df_knn_imputed.loc[missing_mask, 'age'].head(5).values
})
print(comparison)

---
## 3. Outlier Detection and Handling

### 🧠 Interview Concepts

**Detection Methods:**
- **Z-score:** Points > 3 standard deviations from mean
- **IQR (Interquartile Range):** Points outside Q1 - 1.5*IQR or Q3 + 1.5*IQR
- **Isolation Forest:** ML-based anomaly detection

**Handling Strategies:**
- Remove (if errors)
- Cap/Winsorize (clip to boundaries)
- Transform (log, sqrt)
- Keep (if legitimate extreme values)

In [ ]:
# ============================================================
# OUTLIER DETECTION
# ============================================================

def detect_outliers_iqr(df, column):
    """Detect outliers using IQR method."""
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = (df[column] < lower_bound) | (df[column] > upper_bound)
    return outliers, lower_bound, upper_bound

def detect_outliers_zscore(df, column, threshold=3):
    """Detect outliers using Z-score method."""
    mean = df[column].mean()
    std = df[column].std()
    z_scores = np.abs((df[column] - mean) / std)
    return z_scores > threshold

# Analyze outliers in income
print("Outlier Analysis for 'income':")
print("="*50)

outliers_iqr, lb, ub = detect_outliers_iqr(df, 'income')
outliers_zscore = detect_outliers_zscore(df, 'income')

print(f"IQR Method: {outliers_iqr.sum()} outliers (bounds: {lb:.0f} to {ub:.0f})")
print(f"Z-score Method: {outliers_zscore.sum()} outliers (|z| > 3)")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Box plot
axes[0].boxplot(df['income'].dropna())
axes[0].set_title('Income Distribution (with outliers)')
axes[0].set_ylabel('Income')

# Histogram with boundaries
axes[1].hist(df['income'].dropna(), bins=50, edgecolor='black')
axes[1].axvline(lb, color='r', linestyle='--', label=f'Lower bound: {lb:.0f}')
axes[1].axvline(ub, color='r', linestyle='--', label=f'Upper bound: {ub:.0f}')
axes[1].set_title('Income Histogram with IQR Bounds')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# OUTLIER HANDLING
# ============================================================

def cap_outliers(df, column):
    """Cap outliers to IQR boundaries (Winsorization)."""
    _, lower_bound, upper_bound = detect_outliers_iqr(df, column)
    df_capped = df.copy()
    df_capped[column] = df_capped[column].clip(lower=lower_bound, upper=upper_bound)
    return df_capped

# Cap income outliers
df_capped = cap_outliers(df, 'income')

print("Before capping:")
print(f"  Min: {df['income'].min():.0f}, Max: {df['income'].max():.0f}")
print("\nAfter capping:")
print(f"  Min: {df_capped['income'].min():.0f}, Max: {df_capped['income'].max():.0f}")

# Fix invalid age values (negative or > 120)
print("\nFixing invalid age values:")
invalid_ages = (df['age'] < 0) | (df['age'] > 120)
print(f"Invalid ages found: {invalid_ages.sum()}")
df_capped.loc[invalid_ages, 'age'] = np.nan  # Mark as missing to impute later

---
## 4. Encoding Categorical Variables

### 🧠 Interview Concepts

| Encoding | When to Use | Example |
|----------|-------------|--------|
| Label Encoding | Ordinal categories | Education: HS < Bachelor's < Master's |
| One-Hot Encoding | Nominal categories (< 10 unique) | Color: Red, Blue, Green |
| Target Encoding | High cardinality nominal | Zip codes (thousands of values) |

In [ ]:
# ============================================================
# FIX INCONSISTENT CATEGORICAL VALUES FIRST!
# ============================================================

print("Unique values in 'gender' before cleaning:")
print(df['gender'].unique())

# Standardize categorical values
df_clean = df_capped.copy()
df_clean['gender'] = df_clean['gender'].str.lower().str.capitalize()

print("\nUnique values after cleaning:")
print(df_clean['gender'].unique())

In [ ]:
# ============================================================
# ENCODING DEMONSTRATIONS
# ============================================================

print("Encoding Demonstrations:")
print("="*60)

# 1. Label Encoding (for binary or ordinal)
print("\n1. Label Encoding (Gender - Binary):")
le = LabelEncoder()
gender_encoded = le.fit_transform(df_clean['gender'])
print(f"   Mapping: {dict(zip(le.classes_, range(len(le.classes_))))}")

# 2. Ordinal Encoding (for ordered categories)
print("\n2. Ordinal Encoding (Contract Type - Ordered):")
contract_order = ['Month-to-month', 'One year', 'Two year']
oe = OrdinalEncoder(categories=[contract_order])
contract_encoded = oe.fit_transform(df_clean[['contract_type']])
print(f"   Order: {contract_order}")
print(f"   Encoded: {np.unique(contract_encoded)}")

# 3. One-Hot Encoding (for nominal categories)
print("\n3. One-Hot Encoding (Payment Method - Nominal):")
ohe = OneHotEncoder(sparse_output=False, drop='first')  # drop first to avoid multicollinearity
payment_encoded = ohe.fit_transform(df_clean[['payment_method']])
print(f"   Original categories: {df_clean['payment_method'].unique()}")
print(f"   Encoded shape: {payment_encoded.shape}")
print(f"   Feature names: {ohe.get_feature_names_out()}")

---
## 5. Feature Scaling

### 🧠 Interview Concepts

| Scaler | Formula | When to Use |
|--------|---------|-------------|
| StandardScaler | (x - mean) / std | Most algorithms, assumes normal distribution |
| MinMaxScaler | (x - min) / (max - min) | Neural networks, bounded outputs |
| RobustScaler | (x - median) / IQR | Data with outliers |

In [ ]:
# ============================================================
# SCALING COMPARISON
# ============================================================

# Prepare sample data
sample_data = df_clean[['income']].dropna()

# Apply different scalers
standard_scaled = StandardScaler().fit_transform(sample_data)
minmax_scaled = MinMaxScaler().fit_transform(sample_data)
robust_scaled = RobustScaler().fit_transform(sample_data)

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

axes[0, 0].hist(sample_data, bins=50, edgecolor='black')
axes[0, 0].set_title('Original Data')

axes[0, 1].hist(standard_scaled, bins=50, edgecolor='black')
axes[0, 1].set_title('StandardScaler (z-score)')

axes[1, 0].hist(minmax_scaled, bins=50, edgecolor='black')
axes[1, 0].set_title('MinMaxScaler (0 to 1)')

axes[1, 1].hist(robust_scaled, bins=50, edgecolor='black')
axes[1, 1].set_title('RobustScaler (robust to outliers)')

plt.tight_layout()
plt.show()

print("Scaling Statistics:")
print(f"Original - Mean: {sample_data.values.mean():.2f}, Std: {sample_data.values.std():.2f}")
print(f"StandardScaler - Mean: {standard_scaled.mean():.2f}, Std: {standard_scaled.std():.2f}")
print(f"MinMaxScaler - Min: {minmax_scaled.min():.2f}, Max: {minmax_scaled.max():.2f}")

---
## 6. Handling Imbalanced Data

### 🧠 Interview Favorite!

**Techniques:**
1. **Resampling:**
   - Oversampling minority class (SMOTE)
   - Undersampling majority class
2. **Algorithm-level:**
   - Class weights
   - Cost-sensitive learning
3. **Threshold adjustment:**
   - Change classification threshold from 0.5

In [ ]:
# ============================================================
# HANDLING IMBALANCED DATA
# ============================================================

# Check class distribution
print("Class Distribution:")
print(df_clean['churn'].value_counts())
print(f"\nImbalance ratio: {df_clean['churn'].value_counts()[0] / df_clean['churn'].value_counts()[1]:.2f}:1")

# Visualize
fig, ax = plt.subplots(figsize=(6, 4))
df_clean['churn'].value_counts().plot(kind='bar', ax=ax)
ax.set_title('Class Distribution')
ax.set_xlabel('Churn')
ax.set_ylabel('Count')
ax.set_xticklabels(['No Churn (0)', 'Churn (1)'], rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CLASS WEIGHTS (Most common in interviews)
# ============================================================

from sklearn.utils.class_weight import compute_class_weight

# Compute class weights
classes = np.unique(df_clean['churn'])
weights = compute_class_weight('balanced', classes=classes, y=df_clean['churn'])
class_weights = dict(zip(classes, weights))

print("Computed Class Weights:")
print(class_weights)

# Use in model training
print("\nUsage in RandomForest:")
print("rf = RandomForestClassifier(class_weight='balanced')")
print("# or")
print(f"rf = RandomForestClassifier(class_weight={class_weights})")

---
## 7. ⚠️ Data Leakage (CRITICAL INTERVIEW TOPIC)

### What is Data Leakage?
Using information during training that would not be available at prediction time.

### Common Sources:
1. **Target leakage:** Features that contain target information
2. **Train-test contamination:** Fitting preprocessing on full data before split
3. **Temporal leakage:** Using future data to predict past

### The Golden Rule:
> **Fit preprocessing ONLY on training data, transform both train and test**

In [ ]:
# ============================================================
# DATA LEAKAGE EXAMPLES
# ============================================================

print("❌ WRONG: Scaling before split (causes leakage)")
print("="*50)
print("""
# DON'T DO THIS!
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # Fit on ALL data
X_train, X_test = train_test_split(X_scaled)  # Split after
""")

print("\n✅ CORRECT: Split first, then scale")
print("="*50)
print("""
# DO THIS!
X_train, X_test = train_test_split(X)  # Split first
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # Fit only on train
X_test_scaled = scaler.transform(X_test)  # Transform test
""")

In [ ]:
# ============================================================
# DEMONSTRATION OF LEAKAGE IMPACT
# ============================================================

from sklearn.datasets import make_classification

# Create simple dataset
X, y = make_classification(n_samples=500, n_features=20, random_state=42)

# WRONG WAY (with leakage)
scaler_wrong = StandardScaler()
X_scaled_wrong = scaler_wrong.fit_transform(X)  # Fit on ALL data
X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X_scaled_wrong, y, test_size=0.2, random_state=42
)

# RIGHT WAY (no leakage)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y, test_size=0.2, random_state=42
)
scaler_right = StandardScaler()
X_train_r = scaler_right.fit_transform(X_train_r)  # Fit only on train
X_test_r = scaler_right.transform(X_test_r)  # Transform test

# Compare
print("Comparison (subtle but important difference):")
print(f"\nWrong way - Test mean: {X_test_w.mean():.6f}")
print(f"Right way - Test mean: {X_test_r.mean():.6f}")
print("\nIn real scenarios with complex pipelines, leakage can cause")
print("significant overestimation of model performance!")

---
## 8. Building Reusable Preprocessing Pipelines

**Why Pipelines?**
1. Prevent data leakage
2. Reproducible preprocessing
3. Easy to deploy
4. Works with cross-validation

In [ ]:
# ============================================================
# COMPLETE PREPROCESSING PIPELINE
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Prepare the data
df_pipeline = df_clean.copy()
df_pipeline = df_pipeline.dropna(subset=['churn'])  # Remove rows with missing target

# Define feature groups
numerical_features = ['age', 'income', 'tenure_months', 'monthly_charges', 'total_charges']
categorical_features = ['contract_type', 'payment_method', 'gender']

# Split data FIRST
X = df_pipeline[numerical_features + categorical_features]
y = df_pipeline['churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")

In [ ]:
# ============================================================
# CREATE PIPELINE
# ============================================================

# Numerical pipeline
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

# Combine into ColumnTransformer
preprocessor = ColumnTransformer([
    ('numerical', numerical_pipeline, numerical_features),
    ('categorical', categorical_pipeline, categorical_features)
])

# Full pipeline with model
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=100, 
        class_weight='balanced',
        random_state=42
    ))
])

print("Pipeline created successfully!")
print("\nPipeline structure:")
print(full_pipeline)

In [ ]:
# ============================================================
# TRAIN AND EVALUATE
# ============================================================

# Fit the entire pipeline
full_pipeline.fit(X_train, y_train)

# Predictions
y_pred = full_pipeline.predict(X_test)
y_prob = full_pipeline.predict_proba(X_test)[:, 1]

# Evaluation
print("Model Performance:")
print("="*50)
print(classification_report(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

In [ ]:
# ============================================================
# CROSS-VALIDATION WITH PIPELINE (No Leakage!)
# ============================================================

# Cross-validation automatically handles the fit/transform correctly
cv_scores = cross_val_score(
    full_pipeline, X, y, 
    cv=5, 
    scoring='roc_auc'
)

print("5-Fold Cross-Validation (ROC-AUC):")
print(f"Scores: {cv_scores.round(4)}")
print(f"Mean: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

In [ ]:
# ============================================================
# SAVE PIPELINE FOR DEPLOYMENT
# ============================================================

import joblib

# Save the entire pipeline
joblib.dump(full_pipeline, 'churn_prediction_pipeline.joblib')
print("Pipeline saved to 'churn_prediction_pipeline.joblib'")

# Load and use
loaded_pipeline = joblib.load('churn_prediction_pipeline.joblib')
sample_prediction = loaded_pipeline.predict(X_test.head(1))
print(f"\nSample prediction: {sample_prediction[0]}")

# Clean up
import os
os.remove('churn_prediction_pipeline.joblib')

---
## 📝 Week 3 Interview Questions

1. **How do you handle missing values?**
   - First understand WHY they're missing (MCAR, MAR, MNAR)
   - Simple: mean/median for numerical, mode for categorical
   - Advanced: KNN imputation, model-based imputation

2. **How do you detect outliers?**
   - Statistical: Z-score (> 3σ), IQR method
   - Visual: Box plots, scatter plots
   - ML-based: Isolation Forest

3. **When would you use One-Hot vs Label Encoding?**
   - One-Hot: Nominal categories (no order)
   - Label/Ordinal: Ordered categories
   - Target encoding: High cardinality

4. **What is data leakage and how do you prevent it?**
   - Using info not available at prediction time
   - Prevention: Split first, then preprocess
   - Use pipelines!

5. **How do you handle imbalanced data?**
   - Resampling: SMOTE, undersampling
   - Class weights
   - Choose appropriate metrics (F1, ROC-AUC)

6. **Why use pipelines?**
   - Prevent data leakage
   - Reproducibility
   - Easy deployment
   - Works with cross-validation

---

## ✅ Week 3 Checklist

- [ ] Analyze and handle missing values
- [ ] Detect and handle outliers
- [ ] Encode categorical variables correctly
- [ ] Scale features appropriately
- [ ] Handle imbalanced datasets
- [ ] Understand and prevent data leakage
- [ ] Build reusable preprocessing pipelines

---

**Next: Week 4 - End-to-End Project** 🚀